# Preparación de la vista minable para SPSS Statistics
## TPI Ciencia de Datos — DS Airlines

Este notebook tiene un único propósito: tomar el dataset crudo `Vuelos.xlsx`, aplicar las **mismas transformaciones y la misma partición 70/30** definidas en la Fase 3, y exportar el resultado en formato listo para abrir en **SPSS Statistics**.

Se utiliza para los modelos de la Fase 4 que se implementan en Statistics (Análisis Discriminante y Regresión Logística), de modo que trabajen exactamente sobre los mismos datos que el Árbol de Decisión y KNN implementados en Python.

Se exporta en tres formatos (`.sav`, `.xlsx`, `.csv`) para tener alternativas de carga; el `.sav` es el formato nativo de SPSS y el más confiable.

---

## 1. Librerías y carga del dataset crudo

In [ ]:
import pandas as pd

RANDOM_STATE = 42

# Subí Vuelos.xlsx al entorno de Colab antes de correr esta celda
# (panel izquierdo -> carpeta -> subir archivo), o montá Google Drive.
df = pd.read_excel('Vuelos.xlsx')
print(f'Dataset crudo: {df.shape}')
df.head()

Dataset crudo: (15000, 15)


,id_vuelo,aeropuerto_origen,aeropuerto_destino,hora_salida_programada,dia_semana,distancia_vuelo,condiciones_climaticas,congestion_aerea,tipo_avion,ocupacion_vuelo,temporada_alta,puerta_embarque,visibilidad,tiempo_estimado_vuelo,demora
0,10002,GIG,GRU,09:30,Miércoles,360,Despejado,Baja,Boeing 737,52.8,False,45,12.2,57,1
1,10003,TUC,EZE,11:00,Sábado,1230,Despejado,Alta,Embraer E195,54.4,True,11,10.7,132,1
2,10005,MEX,PTY,08:15,Viernes,2280,Niebla,Media,Boeing 757,48.7,True,7,1.0,199,0
3,10006,POA,GRU,06:45,Martes,1110,Despejado,Baja,Airbus A320neo,68.0,False,15,15.8,109,1
4,10008,PTY,EZE,11:30,Domingo,5100,Despejado,Baja,Airbus A330,92.0,True,36,23.0,421,0


## 2. Eliminación de columnas (decisiones de la Fase 3)

Se descartan el identificador técnico y las variables que el análisis bivariante mostró sin poder discriminante (aeropuertos, tipo de avión, puerta de embarque) y `tiempo_estimado_vuelo` por su colinealidad casi perfecta con `distancia_vuelo` (r = 0,999).

In [ ]:
# id_vuelo se conserva aparte para trazabilidad
id_vuelo = df['id_vuelo'].copy()

cols_excluidas = ['id_vuelo', 'aeropuerto_origen', 'aeropuerto_destino',
                  'tipo_avion', 'puerta_embarque', 'tiempo_estimado_vuelo']
df = df.drop(columns=cols_excluidas)

print(f'Tras eliminar columnas: {df.shape}')
print(f'Columnas restantes: {list(df.columns)}')

Tras eliminar columnas: (15000, 9)
Columnas restantes: ['hora_salida_programada', 'dia_semana', 'distancia_vuelo', 'condiciones_climaticas', 'congestion_aerea', 'ocupacion_vuelo', 'temporada_alta', 'visibilidad', 'demora']


## 3. Transformaciones de la Fase 3

- `hora_salida_programada`: de texto "HH:MM" a número decimal.
- `temporada_alta`: de booleano a 0/1.
- Categóricas (`dia_semana`, `condiciones_climaticas`, `congestion_aerea`): codificación one-hot con categoría de referencia (Lunes, Despejado, Baja).

No se aplica escalamiento min-max: el Análisis Discriminante es invariante a transformaciones lineales y la Regresión Logística tampoco lo requiere.

In [ ]:
# Hora a decimal: "09:30" -> 9.5
def hora_a_decimal(h):
    if isinstance(h, str):
        hh, mm = h.split(':')
        return int(hh) + int(mm) / 60
    return h

df['hora_salida_programada'] = df['hora_salida_programada'].apply(hora_a_decimal)

# Booleano a 0/1
df['temporada_alta'] = df['temporada_alta'].astype(int)

# One-hot encoding
df = pd.get_dummies(df, columns=['dia_semana', 'condiciones_climaticas', 'congestion_aerea'],
                    drop_first=False, dtype=int)

# Eliminar categorías de referencia
referencias = ['dia_semana_Lunes', 'condiciones_climaticas_Despejado', 'congestion_aerea_Baja']
df = df.drop(columns=referencias)

print(f'Vista minable: {df.shape}')
print(f'Columnas: {list(df.columns)}')

Vista minable: (15000, 18)
Columnas: ['hora_salida_programada', 'distancia_vuelo', 'ocupacion_vuelo', 'temporada_alta', 'visibilidad', 'demora', 'dia_semana_Domingo', 'dia_semana_Jueves', 'dia_semana_Martes', 'dia_semana_Miércoles', 'dia_semana_Sábado', 'dia_semana_Viernes', 'condiciones_climaticas_Lluvia', 'condiciones_climaticas_Niebla', 'condiciones_climaticas_Nublado', 'condiciones_climaticas_Tormenta', 'congestion_aerea_Alta', 'congestion_aerea_Media']


## 4. Partición 70/30 estratificada (semilla 42)

Se reproduce la misma partición usada en el notebook de Árbol y KNN. Se agrega la columna `sel_train` (1 = entrenamiento, 0 = prueba), que en SPSS se usa como **variable de selección** para entrenar con el 70 % y evaluar sobre el 30 % restante.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['demora'])
y = df['demora']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

# Reconstruir el dataset completo con la marca de partición
vista_spss = X.copy()
vista_spss['demora'] = y
vista_spss['sel_train'] = 0
vista_spss.loc[X_train.index, 'sel_train'] = 1

print(f'Vista para SPSS: {vista_spss.shape}')
print(vista_spss['sel_train'].value_counts())   # 10500 train / 4500 test
print(f'\nColumnas finales:\n{list(vista_spss.columns)}')

Vista para SPSS: (15000, 19)
sel_train
1    10500
0     4500
Name: count, dtype: int64

Columnas finales:
['hora_salida_programada', 'distancia_vuelo', 'ocupacion_vuelo', 'temporada_alta', 'visibilidad', 'dia_semana_Domingo', 'dia_semana_Jueves', 'dia_semana_Martes', 'dia_semana_Miércoles', 'dia_semana_Sábado', 'dia_semana_Viernes', 'condiciones_climaticas_Lluvia', 'condiciones_climaticas_Niebla', 'condiciones_climaticas_Nublado', 'condiciones_climaticas_Tormenta', 'congestion_aerea_Alta', 'congestion_aerea_Media', 'demora', 'sel_train']


## 5. Limpieza de nombres y exportación

Los nombres de variable en SPSS no aceptan acentos. Se reemplazan antes de exportar (p. ej. `dia_semana_Miércoles` -> `dia_semana_Miercoles`).

In [ ]:
# Quitar acentos de los nombres de columna
vista_export = vista_spss.copy()
vista_export.columns = (vista_export.columns
                        .str.replace('á','a').str.replace('é','e').str.replace('í','i')
                        .str.replace('ó','o').str.replace('ú','u'))

print('Nombres listos para SPSS:')
print(list(vista_export.columns))

Nombres listos para SPSS:
['hora_salida_programada', 'distancia_vuelo', 'ocupacion_vuelo', 'temporada_alta', 'visibilidad', 'dia_semana_Domingo', 'dia_semana_Jueves', 'dia_semana_Martes', 'dia_semana_Miercoles', 'dia_semana_Sabado', 'dia_semana_Viernes', 'condiciones_climaticas_Lluvia', 'condiciones_climaticas_Niebla', 'condiciones_climaticas_Nublado', 'condiciones_climaticas_Tormenta', 'congestion_aerea_Alta', 'congestion_aerea_Media', 'demora', 'sel_train']


### 5.1 Formato .sav (nativo de SPSS — el más confiable)

In [ ]:
# pyreadstat escribe el formato propio de SPSS, sin problemas de importación
!pip install pyreadstat -q
import pyreadstat

pyreadstat.write_sav(vista_export, 'vista_minable_spss.sav')
print('Generado: vista_minable_spss.sav')

Generado: vista_minable_spss.sav


### 5.2 Formatos .xlsx y .csv (alternativas)

In [ ]:
vista_export.to_excel('vista_minable_spss.xlsx', index=False, engine='openpyxl')
vista_export.to_csv('vista_minable_spss.csv', index=False)
print('Generados: vista_minable_spss.xlsx  y  vista_minable_spss.csv')

Generados: vista_minable_spss.xlsx  y  vista_minable_spss.csv


## 6. Descarga

Descargá el formato que prefieras. Recomendación: empezá por el `.sav`. Si por algún motivo no lo podés abrir, probá el `.csv` (Archivo → Importar datos → Texto/CSV en SPSS).

In [ ]:
from google.colab import files

files.download('vista_minable_spss.sav')
# files.download('vista_minable_spss.csv')   # descomentá si necesitás el CSV
# files.download('vista_minable_spss.xlsx')  # descomentá si necesitás el Excel